# Homework - Grover MaxCut

The places where you have enter code are marked with `# YOUR CODE HERE`.

In [1]:
import cirq
from cirq import H, X, Y, Z, CX, inverse

## Question 1 (4 points)

Write a function, `oracle010`, that implements an oracle that marks the state $|010 \rangle$. The function `oracle010` has

* input: `qq`, a 3-qubit register 
* returns: `None`

The function should append a sequence of gates to `qq` to mark the state $|010\rangle$ only. Don't append any measurements to `qq`.

To help you test the function, we have provided the `grover_diffusion` and `grover` functions.

In [2]:
def oracle010(qq):
    # YOUR CODE HERE
     # Apply X gates to the first and third qubits to transform |010> to |111>
    yield cirq.X(qq[0])
    yield cirq.X(qq[2])
    
    # Apply a controlled-controlled-Z (CCZ) gate to flip the phase of |111>
    yield cirq.H(qq[2])
    yield cirq.CCX(qq[0], qq[1], qq[2])
    yield cirq.H(qq[2])
    
    # Revert the X gates
    yield cirq.X(qq[0])
    yield cirq.X(qq[2])

In [3]:
# visualize your implemented gates
qqTest = cirq.LineQubit.range(3)
circuit = cirq.Circuit()
circuit.append(oracle010(qqTest))
circuit

0: ───X───────@───X───────
              │
1: ───────────@───────────
              │
2: ───X───H───X───H───X───

In [4]:
# To check your solution, we need some to implement grover
def grover_diffusion(qq,n):
    yield H.on_each(*qq)
    yield X.on_each(*qq)
    yield Z(qq[n-1]).controlled_by(*(qq[0:n-1]))
    yield X.on_each(*qq)
    yield H.on_each(*qq)

In [5]:
def grover(trials_number):
    n=3
    qq = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()
    circuit.append(H.on_each(*qq))  

    for i in range(2):
        circuit.append(oracle010(qq))
        circuit.append(grover_diffusion(qq,n))
    circuit.append(cirq.measure(*qq, key='result'))

    # determine the statistics of the measurements
    s = cirq.Simulator() 
    samples = s.run(circuit, repetitions=trials_number)

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    print(counts)
    return counts.get('010')

In [6]:
# run grover to test if your function gives the right answer
grover(100)

Counter({'010': 96, '111': 3, '110': 1})


96

In [7]:
# hidden tests in this cell will be used for grading.

## Question 2 (6 points)

Graph $G$ has 5 vertices and 6 edges: (0,3), (0,4), (1,3), (1,4), (2,3), (2,4).

Write an oracle for the graph $G$ to check whether it admits a valid 2-coloring. 

The function `oracle2` has

* input: `qq`, a 12-qubit register 
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits 0-4 for the vertices, 5-10 for the edges and 11 as the ancilla. 

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation2` functions.

In [8]:
qq = cirq.LineQubit.range(12)
def edge_check(a, b, c):
    yield CX(qq[a], qq[c])
    yield CX(qq[b], qq[c])
def oracle2(qq):
    # YOUR CODE HERE
    # check 0-3 edge and store at 4th qubit
    yield edge_check(0, 3, 5)
    # check 0-4 edge and store at 5th qubit
    yield edge_check(0, 4, 6)
    # check 1-3 edge and store at 6th qubit
    yield edge_check(1, 3, 7)
    # check 1-4 edge and store at 7th qubit
    yield edge_check(1, 4, 8)
    # check 2-3 edge and store at 7th qubit
    yield edge_check(2, 3, 9)
    # check 2-4 edge and store at 7th qubit
    yield edge_check(2, 4, 10)
    # check all edge qubits
    yield X(qq[11]).controlled_by(*(qq[5:11]))

In [9]:
# We need some code so you can check your solution
def oracle_computation2(qq):
    yield oracle2(qq)
    yield Z(qq[11])
    yield inverse(oracle2(qq))  

In [10]:
def grover2(trials_number):    
    import cirq
    from cirq import X, H, Z, inverse, CX
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(12)
    n=5
    
    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation2(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [11]:
#You can use this cell to test your solution
shots=1000
grover2(shots)

Counter({'00011': 470,
         '11100': 430,
         '00110': 9,
         '01110': 8,
         '11010': 7,
         '11111': 7,
         '10101': 6,
         '00111': 6,
         '10100': 5,
         '10011': 4,
         '01010': 4,
         '00010': 4,
         '00000': 4,
         '10000': 4,
         '01100': 3,
         '00100': 3,
         '10010': 3,
         '10001': 3,
         '01101': 2,
         '11110': 2,
         '11101': 2,
         '01111': 2,
         '11000': 2,
         '01011': 2,
         '00001': 2,
         '00101': 1,
         '11011': 1,
         '10110': 1,
         '11001': 1,
         '01001': 1,
         '01000': 1})

In [12]:
# hidden tests in this cell will be used for grading.

## Question 3 (10 points)

Graph $G$ has 4 vertices and 5 edges: (0,1), (0,2), (0,3), (1,2), (1,3)

Write an oracle for the graph $G$ to check whether there exists a coloring with at least 4 edges connecting vertices with different colors.

The function `oracle3` has

* input: `qq`, a 13-qubit register 
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits 
- 0-3 for the vertices,
- 4-8 for the edges,
- 9-11 for the addition (remember we need three qubits here for addition unlike the last question), and
- 12 as the ancilla.

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation3` functions.

In [13]:
qq = cirq.LineQubit.range(13)
def edge_check( a, b, c):
    yield cirq.CNOT(qq[a], qq[c])
    yield cirq.CNOT(qq[b], qq[c])
def oracle3(qq):
    # YOUR CODE HERE
     # YOUR CODE HERE
    yield edge_check(0, 1, 4)
    yield edge_check(0, 2, 5)
    yield edge_check(0, 3, 6)
    yield edge_check(1, 2, 7)
    yield edge_check(1, 3, 8)
    
    # Accumulate edge check results
    yield cirq.CNOT(qq[4], qq[9])
    yield cirq.CCX(qq[5], qq[9], qq[10])
    yield cirq.CNOT(qq[5], qq[9])
    
    yield cirq.CCX(qq[6], qq[10], qq[11])
    yield cirq.CNOT(qq[6], qq[10])
    
    yield cirq.CCX(qq[7], qq[11], qq[12])
    yield cirq.CNOT(qq[7], qq[11])
    
    yield cirq.CCX(qq[8], qq[12], qq[11])  # Corrected index
    
    # Check if at least 4 edges are differently colored
    yield cirq.X(qq[9])
    yield cirq.X(qq[10])
    yield cirq.X(qq[11])
    yield cirq.X(qq[12]).controlled_by(qq[9], qq[10], qq[11])
    # note we don't have to undo the X gates!

In [14]:
# We need some code so you can check your solution
def oracle_computation3(qq):
    yield oracle3(qq)
    yield Z(qq[12])
    yield inverse(oracle3(qq))  

In [15]:
import cirq
from cirq import X, H, Z, inverse, CX, CCX
    
def grover3(trials_number):    
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(13)
    n=4

    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation3(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [16]:
#You can use this cell to test your solution
shots=1000
grover3(shots)

Counter({'0000': 480,
         '1111': 467,
         '1010': 8,
         '1101': 7,
         '1001': 5,
         '0001': 5,
         '0110': 5,
         '0111': 4,
         '1100': 4,
         '0101': 4,
         '1000': 3,
         '1011': 2,
         '0010': 2,
         '0011': 2,
         '0100': 1,
         '1110': 1})

In [ ]:
# hidden tests in this cell will be used for grading.